# Polymarket Optimizer — Parameter Grid Search

Coarse grid search over each strategy's two main parameters at a fixed 50ms
latency. ConvergenceNo is the one reference strategy shipped with the repo
(in-sample positive but **overfit** — see `docs/FINDINGS.md`); add your own
`Strategy` subclasses to `STRATEGY_SPECS` to sweep them the same way.

> Opens DuckDB in **read-only** mode — no lock conflict with `run_backtest.ipynb`, no re-ingestion, no OOM.

## 0. Setup

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings('ignore')

# Resolve backtest package — same logic as run_backtest.ipynb
for _p in [
    pathlib.Path.cwd().parent,
    pathlib.Path.cwd() / 'research',
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent.parent,
]:
    if (_p / 'backtest' / '__init__.py').exists():
        sys.path.insert(0, str(_p))
        break

import numpy as np
import pandas as pd
import plotly.graph_objects as go

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

from backtest.data_loader import DataLoader
from backtest.optimizer import CoarseGridSearch, BUCKET_B_LATENCY_MS
from backtest.strategies import ConvergenceNo

print('Imports OK.')

## 1. Load Data

Opens the DuckDB file in **`read_only=True`** mode. Multiple readers can attach simultaneously — no exclusive lock needed, no re-ingestion of ticks, no OOM.

**Prerequisite:** `run_backtest.ipynb` cell 2 must have run at least once to build the `ticks` table in `/tmp/polymarket_backtest.duckdb`.

In [ ]:
# Auto-detect data directory (same pattern as run_backtest.ipynb)
for _base in [
    pathlib.Path.cwd(),
    pathlib.Path.cwd().parent,
    pathlib.Path.cwd().parent.parent,
]:
    if (_base / 'data' / 'parquet').exists():
        _pq   = _base / 'data' / 'parquet'
        _meta = _base / 'data' / 'market_metadata.csv'
        break

# read_only=True: attach to the existing file without acquiring a write lock.
# _load() returns immediately in read-only mode — no staleness check, no CREATE TABLE.
loader = DataLoader(parquet_dir=_pq, metadata_csv=_meta, read_only=True)
_ = loader.con  # open connection now so the cost is visible here, not in sweep timings

s = loader.summary()
print(f"Ticks:   {s['total_ticks']:>14,}")
print(f"Trades:  {s['trades']:>14,}")
print(f"Markets: {s['markets']:>14,}")
print(f"Tokens:  {s['tokens']:>14,}")
print(f"Span:    {s['hours']:>13.1f}h")

## 2. Configuration

In [ ]:
STARTING_CAPITAL = 10_000   # capital gate; set to 1000 for realistic ROC

# Add your own Strategy subclasses here to sweep them alongside ConvergenceNo.
STRATEGY_SPECS = [
    dict(
        cls=ConvergenceNo,
        param1='threshold',       p1_vals=[0.25, 0.30, 0.35, 0.40, 0.45],
        param2='exit_threshold',  p2_vals=[0.75, 0.80, 0.85, 0.90, 0.95],
        fixed={'size': 10.0},
    ),
]

print(f'Capital gate:  ${STARTING_CAPITAL:,}')
print(f'Strategies:    {len(STRATEGY_SPECS)} × 5×5 grid at {BUCKET_B_LATENCY_MS}ms')

---
## 3. Grid Search

5×5 sweep at **50ms latency**. `tqdm` progress bars show live per-cell PnL.

**What to look for:** green clusters with ★ plateau markers — cells within 10% of global max that have ≥2 green neighbours. Isolated spikes are noise; plateaus are robust.

In [ ]:
grid_results = {}   # cls.name → GridResult
gs = CoarseGridSearch()

for spec in STRATEGY_SPECS:
    cls = spec['cls']
    result = gs.run(
        strategy_cls=cls,
        param1=spec['param1'],  p1_vals=spec['p1_vals'],
        param2=spec['param2'],  p2_vals=spec['p2_vals'],
        fixed_kwargs=spec['fixed'],
        loader=loader,
        starting_capital=STARTING_CAPITAL,
    )
    grid_results[cls.name] = result

### 3a. Heatmaps — Total PnL

RdYlGn colorscale, zero-centred on white. ★ = plateau (robust optimum).

In [ ]:
if not grid_results:
    print('No results yet — run the cell above first.')
else:
    from IPython.display import display, HTML

    for result in grid_results.values():
        fig = gs.plot_heatmap(result, metric='total_pnl')
        fig.update_layout(template='plotly_dark')
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

### 3b. Heatmaps — Return on Capital

In [ ]:
if not grid_results:
    print('No results yet.')
else:
    from IPython.display import display, HTML

    for result in grid_results.values():
        fig = gs.plot_heatmap(result, metric='roc')
        fig.update_layout(template='plotly_dark')
        display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

---
## 4. Summary — Best Cell Per Strategy

In [ ]:
if not grid_results:
    print('Run the grid search first.')
else:
    rows = []
    for name, result in grid_results.items():
        z = result.pnl_matrix
        i, j = np.unravel_index(np.nanargmax(z), z.shape)
        rows.append({
            'strategy':           name,
            result.param1_name:   result.param1_values[i],
            result.param2_name:   result.param2_values[j],
            'best_pnl':           round(float(z[i, j]), 4),
            'grid_max':           round(float(np.nanmax(z)), 4),
            'grid_min':           round(float(np.nanmin(z)), 4),
            'n_positive_cells':   int((z > 0).sum()),
        })
    display(
        pd.DataFrame(rows)
          .sort_values('best_pnl', ascending=False)
          .reset_index(drop=True)
    )

---
## 5. Drill-Down — Zoom Into a Plateau

After spotting a plateau, edit `DRILL_P1_VALS` / `DRILL_P2_VALS` with finer values and re-run just this cell.

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────
DRILL_CLS     = ConvergenceNo
DRILL_P1      = 'threshold'
DRILL_P1_VALS = [0.38, 0.40, 0.42, 0.44, 0.46]
DRILL_P2      = 'exit_threshold'
DRILL_P2_VALS = [0.90, 0.92, 0.94, 0.96, 0.98]
DRILL_FIXED   = {'size': 10.0}
# ─────────────────────────────────────────────────────────────────────────

from IPython.display import display, HTML

drill = gs.run(
    strategy_cls=DRILL_CLS,
    param1=DRILL_P1,    p1_vals=DRILL_P1_VALS,
    param2=DRILL_P2,    p2_vals=DRILL_P2_VALS,
    fixed_kwargs=DRILL_FIXED,
    loader=loader,
    starting_capital=STARTING_CAPITAL,
)

fig = gs.plot_heatmap(drill, metric='total_pnl')
fig.update_layout(title=f'{DRILL_CLS.__name__} — Drill-Down', template='plotly_dark')
display(HTML(fig.to_html(full_html=False, include_plotlyjs='cdn')))

display(drill.to_dataframe().sort_values('total_pnl', ascending=False).head(10))